In [1]:
import pandas as pd

data1 = pd.read_csv("image_code_output.csv")
data1.head()

,code,image_path
0,271002935.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
1,10028269.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
2,10028268.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
3,10025542.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
4,10027276.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...


In [2]:
data2 = pd.read_csv("AI ML Task Sheet.csv")
data2 = data2.loc[:, ~data2.columns.str.startswith('Unnamed')]
data2

,date,code,qty,rate
0,2026-04-22 14:50:52,500001.0,16.0,1296.0
1,2026-04-22 14:50:52,500001.0,4.0,1295.0
2,2026-04-22 14:50:52,500001.0,16.0,1295.0
3,2026-04-22 14:50:52,500001.0,4.0,1295.0
4,2026-04-22 14:50:52,10029028.0,4.0,1250.0
...,...,...,...,...
994,NaN,NaN,NaN,NaN
995,NaN,NaN,NaN,NaN
996,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN


In [3]:
merged_data = pd.merge(data1, data2, on='code', how='inner')
merged_data_cleaned = merged_data.dropna()

In [4]:
merged_data_cleaned['code'] = merged_data_cleaned['code'].astype('Int64')
merged_data_cleaned['qty'] = merged_data_cleaned['qty'].astype('Int64')

In [6]:
df = merged_data_cleaned.copy()
df = df.groupby('code').agg({
    'image_path': 'first',
    'qty': 'sum',
    'rate': 'first'
}).reset_index()

In [16]:
df

,code,image_path,qty,rate
0,500001,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,1032,1296.0
1,10016728,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,24,875.0
2,10019275,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,56,1650.0
3,10021130,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,32,875.0
4,10021131,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,28,875.0
...,...,...,...,...
121,10029443,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0
122,10029444,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0
123,10029447,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0
124,10029448,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0


In [8]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor


In [ ]:
IMG_SIZE = 224

base_model = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)
base_model.trainable = False

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

N_AUG = 5

In [ ]:

features_list = []
rates_list    = []
targets_list  = []

rate_max = df['rate'].max()

for i in range(len(df)):
    path = df.loc[i, 'image_path']
    img  = cv2.imread(path)
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    rate_norm = df.loc[i, 'rate'] / rate_max

    img_input = preprocess_input(img.astype(np.float32))
    feat = base_model.predict(np.expand_dims(img_input, 0), verbose=0)
    features_list.append(feat.mean(axis=(1, 2)).flatten())
    rates_list.append(rate_norm)
    targets_list.append(df.loc[i, 'qty'])

    img_tensor = tf.cast(img, tf.float32)[tf.newaxis]
    for _ in range(N_AUG):
        aug_img = augment(img_tensor, training=True).numpy()[0]
        aug_img = preprocess_input(aug_img)
        feat = base_model.predict(np.expand_dims(aug_img, 0), verbose=0)
        features_list.append(feat.mean(axis=(1, 2)).flatten())
        rates_list.append(rate_norm)
        targets_list.append(df.loc[i, 'qty'])

X_img = np.array(features_list)                        
X_rate = np.array(rates_list).reshape(-1, 1)            
X = np.hstack([X_img, X_rate])                     
y = np.log1p(np.array(targets_list))

print(f"Dataset size after augmentation: {X.shape[0]} samples")


Dataset size after augmentation: 756 samples


In [ ]:

model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.75,
    colsample_bytree=0.5,
    reg_lambda=3.0,
    reg_alpha=1.0,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
)


In [ ]:

n_orig  = len(df)
n_aug_each = N_AUG + 1
groups = np.repeat(np.arange(n_orig), n_aug_each)

gkf  = GroupKFold(n_splits=5)
maes = []

for train_idx, val_idx in gkf.split(X, y, groups):
    model.fit(X[train_idx], y[train_idx])
    preds  = np.expm1(model.predict(X[val_idx]))
    actual = np.expm1(y[val_idx])
    maes.append(mean_absolute_error(actual, preds))

print(f"CV MAE: {np.mean(maes):.2f}")

model.fit(X, y)


CV MAE: 19.92


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [ ]:

def predict_sales(image_path, rate_value):
    img = cv2.imread(image_path)
    if img is None:
        return "Invalid image path"

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img.astype(np.float32))

    feat = base_model.predict(np.expand_dims(img, 0), verbose=0)
    img_vec = feat.mean(axis=(1, 2)).flatten()

    rate_norm = np.array([rate_value / rate_max])
    X_input   = np.hstack([img_vec, rate_norm]).reshape(1, -1)

    pred = np.expm1(model.predict(X_input)[0])
    return int(round(max(0, pred)))


In [ ]:

# Testing
print(predict_sales(df['image_path'][2], 1650.0))

48


In [ ]:
import joblib, json, os

os.makedirs("saved_model", exist_ok=True)

base_model.save("saved_model/mobilenet_extractor.h5")

joblib.dump(model, "saved_model/xgb_demand_model.pkl")

with open("saved_model/meta.json", "w") as f:
    json.dump({"rate_max": float(rate_max)}, f)

print("Saved! Files:")
print("  saved_model/mobilenet_extractor.h5")
print("  saved_model/xgb_demand_model.pkl")
print("  saved_model/meta.json")


Saved! Files:
  saved_model/mobilenet_extractor.h5
  saved_model/xgb_demand_model.pkl
  saved_model/meta.json
